# Servidor de inferencia del peaje inteligente (RunPod)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito

Levanta **un solo endpoint** (FastAPI) que recibe un frame y devuelve en JSON:
`placa`, `placa_normalizada`, `confianza_ocr`, `categoria`, `ejes`, `pasajeros` y `latencia_ms`.

| Modelo | Qué hace | Pesos |
|---|---|---|
| M1 | Detección de placa + OCR (EasyOCR) + normalización boliviana | `pesos/placas_yolo.pt` o YOLOE zero-shot |
| M2 | Segmentación de instancias: moto, auto, minibus, camion, bus | `pesos/vehiculos_seg.pt` o YOLOE zero-shot |
| M3 | Keypoints de llantas (5 por llanta) → número de ejes | `pesos/llantas_pose.pt` o YOLOE + keypoints geométricos |
| Extra | Conteo de pasajeros visibles (solo cajas, sin rostros) | `yolo11s-seg.pt` (COCO person) |

**Plantilla recomendada:** RunPod PyTorch 2.x (cualquier GPU de 16 GB o más; una RTX A4000/A5000 sobra).
En *Edit Pod → Expose HTTP Ports* agrega **8000**. Igual que en el notebook de la semana 2, la URL pública queda
`https://<POD_ID>-8000.proxy.runpod.net`.

Ejecuta las celdas en orden. Este notebook **no** necesita ninguna API key de RunPod.

## 1. Configuración

In [ ]:
import os, secrets

PUERTO = 8000
# Repositorio público con el proyecto (cámbialo por el tuyo). Si prefieres no usar git, sube el zip de la
# carpeta inferencia/ a /workspace y deja REPO_URL = "".
REPO_URL = "https://github.com/saraquenta/laboratorio3_peajeIA.git"
DIR_PROYECTO = "/workspace/peaje"
DIR_INFERENCIA = f"{DIR_PROYECTO}/inferencia"

# Token que la web enviará en X-API-Token. Define API_TOKEN en las variables de entorno del Pod para que
# no cambie al reiniciar; si no, se genera uno aquí (cópialo a INFERENCE_TOKEN del .env de la web).
API_TOKEN = os.environ.get("API_TOKEN") or secrets.token_urlsafe(24)

POD_ID = os.environ.get("RUNPOD_POD_ID")
URL_PUBLICA = f"https://{POD_ID}-{PUERTO}.proxy.runpod.net" if POD_ID else f"http://localhost:{PUERTO}"
print("INFERENCE_URL   =", URL_PUBLICA)
print("INFERENCE_TOKEN =", API_TOKEN)

## 2. Obtener el código
Opción A (por defecto): `git clone` del repositorio público.
Opción B: sube `inferencia.zip` con el explorador de archivos de Jupyter a `/workspace` y descomenta las líneas de unzip.

In [ ]:
if REPO_URL and not os.path.isdir(DIR_PROYECTO):
    !git clone --depth 1 {REPO_URL} {DIR_PROYECTO}
elif REPO_URL:
    !cd {DIR_PROYECTO} && git pull --ff-only
# --- Opción B ---
# !mkdir -p {DIR_PROYECTO} && unzip -qo /workspace/inferencia.zip -d {DIR_PROYECTO}
!ls {DIR_INFERENCIA}

## 3. Instalación de dependencias
`torch` ya viene en la plantilla de RunPod y **no** se reinstala. La primera vez tarda 2–4 minutos.

In [ ]:
!apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 > /dev/null
!pip install -q -r {DIR_INFERENCIA}/requirements.txt

## 4. Diagnóstico del entorno (Python, torch, CUDA, ultralytics, pesos)

In [ ]:
!cd {DIR_INFERENCIA} && python diagnostico.py

## 5. Pesos propios (opcional pero recomendado)
El servidor busca **exactamente** estos archivos (si falta alguno usa YOLOE zero-shot para ese modelo):

| Archivo | Ruta en el Pod | Lo genera |
|---|---|---|
| `placas_yolo.pt` | `/workspace/peaje/inferencia/pesos/placas_yolo.pt` | `04_M1_placas_ocr.ipynb` |
| `vehiculos_seg.pt` | `/workspace/peaje/inferencia/pesos/vehiculos_seg.pt` | `02_M2_segmentacion_vehiculos.ipynb` |
| `llantas_pose.pt` | `/workspace/peaje/inferencia/pesos/llantas_pose.pt` | `03_M3_keypoints_llantas_ejes.ipynb` |
| `pasajeros_seg.pt` (opcional) | `/workspace/peaje/inferencia/pesos/pasajeros_seg.pt` | `05_M5_pasajeros_segmentacion.ipynb` — si falta se usa `yolo11s-seg.pt` (COCO person) |

**Opción A:** en el explorador de JupyterLab entra a `peaje/inferencia/pesos/` y arrastra los archivos.
**Opción B:** comparte cada archivo de `MyDrive/peaje/pesos/` como *Cualquier persona con el enlace* y pega los enlaces abajo.
La última línea valida que cada `.pt` encaje con lo que espera el servidor (`app/validacion.py`).

In [ ]:
ENLACES_DRIVE = {               # deja '' los que subas a mano
    'placas_yolo.pt': '',
    'vehiculos_seg.pt': '',
    'llantas_pose.pt': '',
    'pasajeros_seg.pt': '',      # opcional (M5)
}
if any(ENLACES_DRIVE.values()):
    !pip install -q gdown
    import gdown
    for nombre, enlace in ENLACES_DRIVE.items():
        if enlace:
            gdown.download(enlace, f"{DIR_INFERENCIA}/pesos/{nombre}", quiet=False, fuzzy=True)
!ls -la {DIR_INFERENCIA}/pesos
!cd {DIR_INFERENCIA} && python diagnostico.py | grep -E "M1|M2|M3|M5"

## 6. Iniciar el servidor
Se lanza como proceso aparte (sigue vivo aunque cierres el notebook). Escucha en `0.0.0.0`
para que el proxy de RunPod lo alcance (con `127.0.0.1` el proxy devuelve 502).

In [ ]:
import subprocess, time, json, urllib.request

entorno = dict(os.environ, API_TOKEN=API_TOKEN, PORT=str(PUERTO))
log = open("/workspace/inferencia.log", "w")
proceso = subprocess.Popen(["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PUERTO), "--workers", "1"],
                           cwd=DIR_INFERENCIA, env=entorno, stdout=log, stderr=subprocess.STDOUT)
print("PID:", proceso.pid, "| log: /workspace/inferencia.log")

# Espera a que los modelos terminen de cargar (la primera vez descarga YOLOE y EasyOCR: 1–3 min)
for i in range(120):
    try:
        salud = json.loads(urllib.request.urlopen(f"http://localhost:{PUERTO}/health", timeout=5).read())
        if salud["estado"] in ("ok", "error"):
            break
    except Exception:
        salud = {"estado": "arrancando"}
    time.sleep(3)
print(json.dumps(salud, indent=2))
print("\nPega esto en el .env de la web (o en Estado → Inferencia):")
print(f"INFERENCE_URL={URL_PUBLICA}")
print(f"INFERENCE_TOKEN={API_TOKEN}")

## 7. Prueba rápida desde el propio Pod

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt

URL_IMAGEN = "https://ultralytics.com/images/bus.jpg"
datos = urllib.request.urlopen(URL_IMAGEN, timeout=30).read()
cab = {"Content-Type": "image/jpeg", "X-API-Token": API_TOKEN}
req = urllib.request.Request(f"http://localhost:{PUERTO}/predict", data=datos, headers=cab)
res = json.loads(urllib.request.urlopen(req, timeout=120).read())
print(json.dumps({k: res[k] for k in ("placa", "placa_normalizada", "confianza_ocr", "categoria", "ejes",
                                      "pasajeros", "latencia_ms")}, indent=2, ensure_ascii=False))

req = urllib.request.Request(f"http://localhost:{PUERTO}/predict/visual", data=datos, headers=cab)
vis = cv2.imdecode(np.frombuffer(urllib.request.urlopen(req, timeout=120).read(), np.uint8), cv2.IMREAD_COLOR)
plt.figure(figsize=(10, 7)); plt.imshow(vis[:, :, ::-1]); plt.axis("off"); plt.show()

## 8. Verificación a través de la URL pública
Si falla, revisa que el puerto 8000 esté en *Expose HTTP Ports* del Pod.

In [ ]:
try:
    print(json.loads(urllib.request.urlopen(f"{URL_PUBLICA}/health", timeout=30).read()))
except Exception as e:
    print("La URL pública aún no responde:", e)

## 9. Ver log / detener el servidor

In [ ]:
!tail -n 30 /workspace/inferencia.log
DETENER = False   # cambia a True para apagar el servidor
if DETENER:
    proceso.terminate(); print("Servidor detenido")